# Laboratorio 10

Juan Ticlia

Jose Huaman

Paulo Miranda

---
## P0: Cargar el Dataset de Rostros

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from rtree import index
import face_recognition
import psycopg2
import time

DATASET_PATH = "/home/tkaos/UTEC/C5/bd2/sol/data/lfw_funneled"

In [ ]:
coleccion = []
for path in glob.iglob(os.path.join(DATASET_PATH, "**", "*.jpg")):
    person = path.split("/")[-2]
    coleccion.append({"person": person, "path": path})

coleccion = pd.DataFrame(coleccion)
print(f"Total de imagenes: {len(coleccion)}")

In [ ]:
def mostrarFotos(coleccion, posiciones):
    plt.figure(figsize=(16, 10))
    i = 0
    for idx in posiciones:
        img = plt.imread(coleccion.path.iloc[idx])
        plt.subplot(4, 4, i + 1)
        plt.imshow(img)
        plt.title(coleccion.person.iloc[idx] + str(img.shape))
        plt.xticks([])
        plt.yticks([])
        i += 1
    plt.tight_layout()
    plt.show()

posiciones = list(range(0, 16))
mostrarFotos(coleccion, posiciones)

---
## Setup BD: Conexion unica a PostgreSQL con pgvector

In [ ]:
conn = psycopg2.connect(
    dbname="postgres",
    user="postgres",
    password="123456",
    host="localhost",
    port="5433"
)
cur = conn.cursor()

cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")
cur.execute("""
    CREATE TABLE IF NOT EXISTS face_embeddings (
        id SERIAL PRIMARY KEY,
        name TEXT,
        path TEXT,
        embedding VECTOR(128)
    );
""")
conn.commit()
print("Extension vector habilitada y tabla face_embeddings lista.")

---
## P1: Generar los vectores caracteristicos (4 pts)

In [ ]:
def generate_face_embeddings(colecion_df, N, cursor):
    rostros_procesados = 0
    for index, row in colecion_df.iterrows():
        if rostros_procesados >= N:
            break
        nombre = row["person"]
        ruta = row["path"]
        try:
            image = face_recognition.load_image_file(ruta)
            face_encodings = face_recognition.face_encodings(image)
            if face_encodings:
                cursor.execute(
                    "INSERT INTO face_embeddings (name, path, embedding) VALUES (%s, %s, %s)",
                    (nombre, ruta, face_encodings[0].tolist())
                )
                rostros_procesados += 1
                print(f"Rostro {rostros_procesados}/{N} guardado: {nombre}")
            else:
                print(f"Sin rostro: {ruta}")
        except Exception as e:
            print(f"Error: {e}")
    conn.commit()
    print(f"\nProcesados {rostros_procesados} rostros.")

In [ ]:
generate_face_embeddings(coleccion, 100, cur)

---
## P2: Busqueda KNN Lineal (3 pts)

In [ ]:
cur.execute("SELECT name, path, embedding::text FROM face_embeddings;")
rows = cur.fetchall()

embeddings_list = []
for name, path, emb_text in rows:
    emb_str = emb_text.replace('[', '').replace(']', '').replace(',', ' ')
    emb_array = np.fromstring(emb_str, sep=' ')
    embeddings_list.append({'person': name, 'path': path, 'embedding': emb_array})

print(f"Embeddings cargados: {len(embeddings_list)}")

In [ ]:
def knn_lineal(q, embeddings_data, k=5, metrica='euclidiana'):
    distancias = []
    for item in embeddings_data:
        v = item['embedding']
        if metrica == 'euclidiana':
            dist = np.linalg.norm(q - v)
        elif metrica == 'coseno':
            cos_sim = np.dot(q, v) / (np.linalg.norm(q) * np.linalg.norm(v))
            dist = 1 - cos_sim
        distancias.append((item['person'], item['path'], dist))
    distancias.sort(key=lambda x: x[2])
    return distancias[:k]

In [ ]:
QUERY_IMAGE = "/home/tkaos/UTEC/C5/bd2/sol/data/mi_consulta.jpg"

if os.path.exists(QUERY_IMAGE):
    img = face_recognition.load_image_file(QUERY_IMAGE)
    encodings = face_recognition.face_encodings(img)
    if encodings:
        q = encodings[0]
        print("Embedding de consulta generado.")

        t0 = time.time()
        r_euc = knn_lineal(q, embeddings_list, k=5, metrica='euclidiana')
        t1 = time.time()
        r_cos = knn_lineal(q, embeddings_list, k=5, metrica='coseno')
        t2 = time.time()

        print(f"KNN Euclidiana - Tiempo: {t1-t0:.4f}s")
        for p, _, d in r_euc:
            print(f"  {p}: {d:.6f}")
        print(f"KNN Coseno - Tiempo: {t2-t1:.4f}s")
        for p, _, d in r_cos:
            print(f"  {p}: {d:.6f}")
    else:
        print("No se detecto rostro.")
else:
    print(f"Imagen no encontrada: {QUERY_IMAGE}")

---
## P3

In [42]:
if os.path.exists(QUERY_IMAGE) and 'q' in locals():
    q_str = "[" + ",".join(str(x) for x in q) + "]"

    cur.execute("DROP INDEX IF EXISTS face_embedding_euc_idx;")
    cur.execute("CREATE INDEX face_embedding_euc_idx ON face_embeddings USING ivfflat (embedding vector_l2_ops) WITH (lists = 100);")
    cur.execute("DROP INDEX IF EXISTS face_embedding_cos_idx;")
    cur.execute("CREATE INDEX face_embedding_cos_idx ON face_embeddings USING ivfflat (embedding vector_cosine_ops) WITH (lists = 100);")
    conn.commit()
    print("Indices IVFFlat creados.")

    cur.execute("SET ivfflat.probes = 10;")

    t0 = time.time()
    cur.execute(f"SELECT id, name, embedding <-> '{q_str}' AS distance FROM face_embeddings ORDER BY embedding <-> '{q_str}' LIMIT 5;")
    r_euc = cur.fetchall()
    t1 = time.time()
    print(f"IVFFlat Euclidiana - Tiempo: {t1-t0:.4f}s")
    for row in r_euc:
        print(f"  {row[1]}: {row[2]:.6f}")

    t0 = time.time()
    cur.execute(f"SELECT id, name, embedding <=> '{q_str}' AS distance FROM face_embeddings ORDER BY embedding <=> '{q_str}' LIMIT 5;")
    r_cos = cur.fetchall()
    t1 = time.time()
    print(f"IVFFlat Coseno - Tiempo: {t1-t0:.4f}s")
    for row in r_cos:
        print(f"  {row[1]}: {row[2]:.6f}")

    cur.execute("DROP INDEX IF EXISTS face_embedding_hnsw_idx;")
    cur.execute("CREATE INDEX face_embedding_hnsw_idx ON face_embeddings USING hnsw (embedding vector_l2_ops) WITH (m = 16, ef_construction = 200);")
    conn.commit()
    print("\nIndice HNSW creado.")

    cur.execute("SET hnsw.ef_search = 50;")
    t0 = time.time()
    cur.execute(f"SELECT id, name, embedding <-> '{q_str}' AS distance FROM face_embeddings ORDER BY embedding <-> '{q_str}' LIMIT 5;")
    r_hnsw = cur.fetchall()
    t1 = time.time()
    print(f"HNSW Euclidiana - Tiempo: {t1-t0:.4f}s")
    for row in r_hnsw:
        print(f"  {row[1]}: {row[2]:.6f}")
else:
    print("Ejecuta primero la celda de consulta en P2.")

Indices IVFFlat creados.
IVFFlat Euclidiana - Tiempo: 0.0005s
  Lee_Yuan-tseh: 0.606601
  Lee_Yuan-tseh: 0.606601
  Lee_Yuan-tseh: 0.606601
  Lee_Yuan-tseh: 0.606601
  Antony_Leung: 0.681518
IVFFlat Coseno - Tiempo: 0.0003s
  Lee_Yuan-tseh: 0.097180
  Lee_Yuan-tseh: 0.097180
  Lee_Yuan-tseh: 0.097180
  Lee_Yuan-tseh: 0.097180
  Antony_Leung: 0.120443

Indice HNSW creado.
HNSW Euclidiana - Tiempo: 0.0007s
  Lee_Yuan-tseh: 0.606601
  Lee_Yuan-tseh: 0.606601
  Lee_Yuan-tseh: 0.606601
  Lee_Yuan-tseh: 0.606601
  Antony_Leung: 0.681518


---
## Cerrar conexion

In [ ]:
cur.close()
conn.close()
print("Conexion cerrada.")